In [23]:
import pandas as pd

# Chemin vers le fichier traité par la collègue
chemin_fichier = "../data/raw/act_metier_juillet.xlsx"

# Lecture de la feuille contenant les actes métier
fichier_excel = pd.ExcelFile(chemin_fichier)

print("Feuilles disponibles dans le fichier :")
print(fichier_excel.sheet_names)

Feuilles disponibles dans le fichier :
['TCD', 'Fichier_acte_metier_20260722095']


In [24]:
# Nom de la feuille contenant les données détaillées
nom_feuille = "Fichier_acte_metier_20260722095"

# Chargement des données
donnees = pd.read_excel(
    chemin_fichier,
    sheet_name=nom_feuille
)

print("Nombre de lignes :", donnees.shape[0])
print("Nombre de colonnes :", donnees.shape[1])

print("\nColonnes disponibles :")
for colonne in donnees.columns:
    print("-", colonne)

Nombre de lignes : 673
Nombre de colonnes : 17

Colonnes disponibles :
- INSEE - Commune
- PDS
- Matricule compteur
- Matricule émetteur
- Scénario
- Date action terrain
- Date reçu SITR
- Date traitement
- Résultat
- Prémonté
- Index mécanique
- Index électronique
- Processus
- Fichier
- Logs
- Source
- Traitement


In [25]:
# Création d'un tableau de synthèse par type de résultat SITR

analyse_resultats = (
    donnees
    .groupby("Résultat")
    .agg(
        Nombre_de_cas=("Résultat", "size"),
        Traitements_renseignes=("Traitement", "count"),
        Nombre_de_traitements_differents=("Traitement", "nunique")
    )
    .reset_index()
)

# Calcul du pourcentage de lignes avec un traitement renseigné
analyse_resultats["Pourcentage_traite"] = (
    analyse_resultats["Traitements_renseignes"]
    / analyse_resultats["Nombre_de_cas"]
    * 100
).round(1)

# Classement du plus fréquent au moins fréquent
analyse_resultats = analyse_resultats.sort_values(
    by="Nombre_de_cas",
    ascending=False
)

# Noms affichés en français
analyse_resultats = analyse_resultats.rename(
    columns={
        "Résultat": "Résultat SITR",
        "Nombre_de_cas": "Nombre de cas",
        "Traitements_renseignes": "Traitements renseignés",
        "Nombre_de_traitements_differents": "Traitements différents",
        "Pourcentage_traite": "Pourcentage traité (%)"
    }
)

analyse_resultats

,Résultat SITR,Nombre de cas,Traitements renseignés,Traitements différents,Pourcentage traité (%)
3,Aucune trame n'a été trouvée dans la plage hor...,349,349,2,100.0
18,Le compteur remonté par l'intervention est dif...,124,0,0,0.0
8,Compteur incompatible - vérifier le type et di...,37,37,9,100.0
5,Ce PDS est déjà associé - Changer le libellé d...,36,34,1,94.4
10,Emetteur déjà associé - Association à vérifier,32,0,0,0.0
6,Ce PDS n'est pas associé - Maintenance non réa...,20,20,4,100.0
4,Ce PDS est absent du patrimoine - Action métie...,19,19,3,100.0
16,Fichier fabricant non reçu,11,11,1,100.0
19,Poids d'impulsion vide - vérifier le diamètre ...,9,9,4,100.0
7,Compteur déjà associé sur un PDS - A vérifier,7,0,0,0.0


In [26]:
# Sélection du rejet étudié
rejet_etudie = donnees[
    donnees["Résultat"]
    .str.contains(
        "Compteur incompatible",
        case=False,
        na=False
    )
].copy()

print("Nombre de cas étudiés :", len(rejet_etudie))

print("\nTraitements renseignés par la collègue :")

traitements_compteur_incompatible = (
    rejet_etudie["Traitement"]
    .value_counts(dropna=False)
    .rename_axis("Traitement")
    .reset_index(name="Nombre de cas")
)

traitements_compteur_incompatible

Nombre de cas étudiés : 37

Traitements renseignés par la collègue :


,Traitement,Nombre de cas
0,EC le 22/07/26 asso,14
1,EC le 24/07/26 asso,8
2,Le 22/07/26 SOP,6
3,Le 22/07/26 correction modèle,4
4,Le 22/07/26 AT,1
5,Le 22/07/26 Correction fabricant,1
6,Le 22/07/26 correction num cptr,1
7,Le 24/07/26 correction fabricant,1
8,Le 24/07/26 SOP,1


In [27]:
# Affichage détaillé des 37 cas "Compteur incompatible"

colonnes_a_analyser = [
    "PDS",
    "Matricule compteur",
    "Matricule émetteur",
    "Scénario",
    "Date action terrain",
    "Date reçu SITR",
    "Prémonté",
    "Index mécanique",
    "Index électronique",
    "Processus",
    "Source",
    "Traitement"
]

colonnes_disponibles = [
    colonne
    for colonne in colonnes_a_analyser
    if colonne in rejet_etudie.columns
]

rejet_etudie[colonnes_disponibles]

,PDS,Matricule compteur,Matricule émetteur,Scénario,Date action terrain,Date reçu SITR,Prémonté,Index mécanique,Index électronique,Processus,Source,Traitement
24,988059036201,D22VA808304,11A5222218601007,Association,2023-12-20 08:28:00,2026-07-17 15:14:00,Non,27.0,NaN,Traitement de l'acte métier,Automatique,Le 22/07/26 AT
29,987259037098,I19JA200893 F01WA370994,2697231919441007,Association,2023-11-14 09:43:00,2026-07-17 15:20:00,Oui,92121.0,0.0,Traitement de l'acte métier,Automatique,Le 22/07/26 SOP
171,985499011164,D23BA109693 ASST,2697240892951007,Association,2025-11-05 08:52:00,2026-07-17 15:21:00,Non,109732.0,NaN,Traitement de l'acte métier,Automatique,Le 22/07/26 SOP
214,983914509966,H26TA606863,11A5261710271007,Association,2026-07-01 15:04:00,2026-07-17 02:01:00,Oui,71.0,-1.0,Traitement de l'acte métier,Automatique,Le 22/07/26 SOP
231,981286471712,D21BA101816,2697251651911007,Association,2026-06-03 05:48:00,2026-07-17 15:22:00,Oui,297767.0,0.0,Traitement de l'acte métier,Automatique,Le 22/07/26 correction modèle
237,983281016078,H26TA616045,11A5267823221007,Association,2026-06-17 07:15:00,2026-07-17 00:00:00,Oui,76.0,12.0,Traitement de l'acte métier,Automatique,Le 22/07/26 SOP
247,985913615738,I19JB023230 F19JB023230,2697251359951007,Association,2026-04-30 10:52:00,2026-07-17 15:22:00,Oui,5683460.0,0.0,Traitement de l'acte métier,Automatique,Le 22/07/26 Correction fabricant
252,983279700446,H22VA215272,11A5251116421007,Association,2026-04-29 08:40:00,2026-07-17 15:18:00,Oui,133721.0,1.0,Traitement de l'acte métier,Automatique,Le 22/07/26 correction modèle
273,983104871962,H26TA532074,11A5261448311007,Association,15/07/26 - 08:49 15/07/2026 06:49:00,2026-07-18 02:01:00,Oui,66.0,0.0,Traitement de l'acte métier,Automatique,EC le 22/07/26 asso
274,983116371960,H26TA532142,11A5267735111007,Association,15/07/26 - 13:27 15/07/2026 11:27:00,2026-07-18 02:01:00,Oui,65.0,1.0,Traitement de l'acte métier,Automatique,EC le 22/07/26 asso


In [28]:
# Analyse des traitements renseignés pour chaque type de rejet

analyse_traitements = (
    donnees.groupby("Résultat")
    .agg(
        Nombre_de_cas=("Résultat", "size"),
        Traitements_renseignes=("Traitement", "count"),
        Nombre_de_traitements_differents=("Traitement", "nunique")
    )
    .reset_index()
)

# Pourcentage de lignes ayant déjà un traitement
analyse_traitements["Taux_traitement_%"] = (
    analyse_traitements["Traitements_renseignes"]
    / analyse_traitements["Nombre_de_cas"]
    * 100
).round(1)

# Tri : rejets les plus fréquents en premier
analyse_traitements = analyse_traitements.sort_values(
    "Nombre_de_cas",
    ascending=False
)

analyse_traitements

,Résultat,Nombre_de_cas,Traitements_renseignes,Nombre_de_traitements_differents,Taux_traitement_%
3,Aucune trame n'a été trouvée dans la plage hor...,349,349,2,100.0
18,Le compteur remonté par l'intervention est dif...,124,0,0,0.0
8,Compteur incompatible - vérifier le type et di...,37,37,9,100.0
5,Ce PDS est déjà associé - Changer le libellé d...,36,34,1,94.4
10,Emetteur déjà associé - Association à vérifier,32,0,0,0.0
6,Ce PDS n'est pas associé - Maintenance non réa...,20,20,4,100.0
4,Ce PDS est absent du patrimoine - Action métie...,19,19,3,100.0
16,Fichier fabricant non reçu,11,11,1,100.0
19,Poids d'impulsion vide - vérifier le diamètre ...,9,9,4,100.0
7,Compteur déjà associé sur un PDS - A vérifier,7,0,0,0.0


In [29]:
import re
import pandas as pd

def nettoyer_traitement(valeur):
    if pd.isna(valeur):
        return pd.NA

    texte = str(valeur).strip().lower()

    # Suppression des dates du type 22/07/26
    texte = re.sub(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", "", texte)

    # Suppression de "le" lorsqu'il reste au début
    texte = re.sub(r"^\s*le\s+", "", texte)

    # Suppression des espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()

    return texte


donnees["Traitement_nettoye"] = donnees["Traitement"].apply(
    nettoyer_traitement
)

# Afficher les traitements nettoyés et leur fréquence
traitements_nettoyes = (
    donnees["Traitement_nettoye"]
    .value_counts(dropna=True)
    .reset_index()
)

traitements_nettoyes.columns = ["Traitement nettoyé", "Nombre de cas"]

traitements_nettoyes

,Traitement nettoyé,Nombre de cas
0,voir nc,348
1,ec le chgt émet,36
2,ec le asso,32
3,en attente,16
4,annulation,11
5,it4us,11
6,sop,7
7,ec le annul,4
8,correction modèle,4
9,correction fabricant,3


In [30]:
def normaliser_traitement(valeur):
    if pd.isna(valeur):
        return pd.NA

    texte = str(valeur).lower().strip()

    # Regroupements explicites
    if "voir nc" in texte:
        return "Voir NC"

    if "chgt émet" in texte:
        return "Changement émetteur"

    if "asso" in texte:
        return "Association"

    if "annul" in texte:
        return "Annulation"

    if "it4us" in texte:
        return "IT4US"

    if "sop" in texte:
        return "SOP"

    if "correction modèle" in texte:
        return "Correction modèle"

    if "correction fabricant" in texte:
        return "Correction fabricant"

    if "correction diam" in texte:
        return "Correction diamètre"

    if "correction num cptr" in texte or "maj num cptr" in texte:
        return "Correction numéro compteur"

    if "maj cptr" in texte:
        return "Mise à jour compteur"

    if texte == "at":
        return "AT"

    if "doublon" in texte:
        return "Doublon"

    if "ok sitr" in texte:
        return "OK SITR"

    if "en attente" in texte or "en attene" in texte:
        return "En attente"

    # On ne devine pas les cas métier non compris
    return "Autre"


donnees["Traitement normalisé"] = (
    donnees["Traitement_nettoye"]
    .apply(normaliser_traitement)
)

repartition_normalisee = (
    donnees["Traitement normalisé"]
    .value_counts(dropna=False)
    .rename_axis("Traitement normalisé")
    .reset_index(name="Nombre de cas")
)

repartition_normalisee

,Traitement normalisé,Nombre de cas
0,Voir NC,348
1,NaN,185
2,Changement émetteur,39
3,Association,34
4,En attente,18
5,Annulation,15
6,IT4US,11
7,SOP,7
8,Correction modèle,4
9,Correction fabricant,3


In [31]:
# Croisement entre le rejet SITR et le traitement réalisé

croisement = pd.crosstab(
    donnees["Résultat"],
    donnees["Traitement normalisé"],
    margins=True,
    margins_name="Total"
)

croisement

Traitement normalisé,AT,Annulation,Association,Autre,Changement émetteur,Correction diamètre,Correction fabricant,Correction modèle,Correction numéro compteur,Doublon,En attente,IT4US,Mise à jour compteur,OK SITR,SOP,Voir NC,Total
Résultat,,,,,,,,,,,,,,,,,
Absence informations dans l'intervention,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4
Affectation KO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
Aucune trame n'a été trouvée dans la plage horaire autorisée,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,348,349
Ce PDS est absent du patrimoine - Action métier impossible - Contacter le support,0,0,0,0,0,0,0,0,0,0,18,0,1,0,0,0,19
Ce PDS est déjà associé - Changer le libellé d'intervention,0,0,0,0,34,0,0,0,0,0,0,0,0,0,0,0,34
Ce PDS n'est pas associé - Maintenance non réalisable,0,11,6,0,3,0,0,0,0,0,0,0,0,0,0,0,20
Compteur incompatible - vérifier le type et diamètre compteur (900002011),1,0,22,0,0,0,2,4,1,0,0,0,0,0,7,0,37
Fichier fabricant non reçu,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,11
L'acte métier porte sur un émetteur différent de celui associé dans SITR,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,3


In [32]:
# Chargement du parc compteurs ODYSSEE
chemin_parc = "../data/raw/Parc_compteur.csv"

parc_compteurs = pd.read_csv(
    chemin_parc,
    sep=";",
    encoding="utf-8",
    low_memory=False
)

print("Nombre de lignes :", parc_compteurs.shape[0])
print("Nombre de colonnes :", parc_compteurs.shape[1])

print("\nColonnes disponibles :")
for colonne in parc_compteurs.columns:
    print("-", colonne)

Nombre de lignes : 361399
Nombre de colonnes : 105

Colonnes disponibles :
- PB
- BANCO
- CONTRAT_RATTACHEMENT
- NATURE_CONTRAT_RATTACHEMENT
- NATURE_SERV_EAU_ASS
- DT_FIN_CONTRAT_CADRE
- COMMUNE
- CODE_CYCLE_SERVICE
- LIBELLE_CYCLE_SERVICE
- CODE_TOURNEE
- LIBELLE_TOURNEE_SERVICE
- TOURNEE_OPALE
- SEQ_TOURNEE
- ID_COMPTEUR
- NUMERO_BADGE
- NUMERO_SERIE
- CARNET_METROLOGIQUE
- FABRICANT
- MODELE
- DIAMETRE
- ANNEE_FABRICATION
- SOLUTION_COMPACTE
- SOLUTION_DEPORTEE
- PASSERELLE_SITE_ISOLE
- DATE_RECEPTION
- DATE_DEPOSE
- MATRICULE_EQUIPEMENT
- ACCESSIBLE
- CODE_EMPLACEMENT
- LIBELLE_EMPLACEMENT
- DETAILS_EMPLACEMENTS
- ID_PDS
- DATE_MISE_EN_SERVICE_PDS
- ETAT_PDS
- DT_SUPPRESSION_PDS
- ETAT_SOURCE_PDS
- MOTIF_FERMETURE_BRANCHEMENT
- REFERENCE_EXTERNE_PDS
- LOGEMENT_VACANT
- GEN_DIV
- ID_PDS_GEN
- RACCORDE_RACCORDABLE
- DATE_RACCORDABILITE
- MO_RES
- NAT_REJ
- REJET
- RESEAU
- ANC_A_FACTURER
- FORAGE
- CODE_PROTECTION
- USAGE
- FLUIDE
- PDS_STRATEGIQUE
- MAT_BRANCH_AV_COMPT
- MAT_AP_COM

In [33]:
# Vérifier le rapprochement entre ACT Métier et le parc ODYSSEE

# On travaille sur une copie pour ne pas modifier les données originales
actes = donnees.copy()
parc = parc_compteurs.copy()

# Nettoyage des identifiants compteur pour rendre la comparaison fiable
actes["Matricule_compteur_nettoye"] = (
    actes["Matricule compteur"]
    .astype("string")
    .str.strip()
    .str.upper()
)

parc["Numero_serie_nettoye"] = (
    parc["NUMERO_SERIE"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Rapprochement :
# Matricule compteur (ACT Métier) <-> NUMERO_SERIE (ODYSSEE)
rapprochement = actes.merge(
    parc[
        [
            "Numero_serie_nettoye",
            "NUMERO_SERIE",
            "FABRICANT",
            "MODELE",
            "DIAMETRE",
            "ID_PDS",
            "ETAT_PDS",
            "MODE_DE_RELEVE"
        ]
    ],
    left_on="Matricule_compteur_nettoye",
    right_on="Numero_serie_nettoye",
    how="left",
    indicator=True
)

# Résultat du rapprochement
nb_total = len(rapprochement)
nb_trouves = (rapprochement["_merge"] == "both").sum()
nb_non_trouves = (rapprochement["_merge"] == "left_only").sum()

print("Nombre de rejets ACT Métier :", nb_total)
print("Compteurs retrouvés dans ODYSSEE :", nb_trouves)
print("Compteurs non retrouvés :", nb_non_trouves)
print(
    "Taux de rapprochement :",
    round(nb_trouves / nb_total * 100, 1),
    "%"
)

Nombre de rejets ACT Métier : 688
Compteurs retrouvés dans ODYSSEE : 358
Compteurs non retrouvés : 330
Taux de rapprochement : 52.0 %


In [34]:
print("ACT MÉTIER")
print(donnees["Matricule compteur"].dropna().head(10).tolist())

print("\nPARC ODYSSEE")
print(parc_compteurs["NUMERO_SERIE"].dropna().head(10).tolist())

ACT MÉTIER
['C22LU377410', 'D22BA100519', 'C25FD008981', 'D23BA066487', 'D22BA065745', 'D19BA172414', 'D22BA053155', 'H25VA980241', 'C23LU076063', 0]

PARC ODYSSEE
['CTR_FICTIF', 'H22VA239424', 'D19BA079538', 'H24VA473587', 'H24VA093590', 'D22BA117321', 'H23VA533364', 'C14FA271535', 'H19VA454580', 'H23VA533305']


In [35]:
# Préparation des PDS pour le rapprochement

actes = donnees.copy()
parc = parc_compteurs.copy()

actes["PDS_nettoye"] = (
    actes["PDS"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

parc["PDS_nettoye"] = (
    parc["ID_PDS"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Vérification du nombre de PDS communs
pds_act = set(actes["PDS_nettoye"].dropna())
pds_ody = set(parc["PDS_nettoye"].dropna())

pds_communs = pds_act.intersection(pds_ody)

print("PDS uniques dans ACT Métier :", len(pds_act))
print("PDS uniques dans ODYSSEE :", len(pds_ody))
print("PDS communs :", len(pds_communs))

print("\nExemples de PDS communs :")
print(list(pds_communs)[:10])

PDS uniques dans ACT Métier : 673
PDS uniques dans ODYSSEE : 361399
PDS communs : 0

Exemples de PDS communs :
[]


In [36]:
import pandas as pd

# Charger le parc compteurs ODYSSEE de juillet
chemin_parc = "../data/raw/Parc_compteur.csv"

parc = pd.read_csv(
    chemin_parc,
    sep=";",
    encoding="utf-8",
    low_memory=False
)

print("Nombre de compteurs dans le parc :", parc.shape[0])
print("Nombre de colonnes :", parc.shape[1])

# Colonnes qui nous intéressent pour le moment
colonnes_utiles = [
    "NUMERO_SERIE",
    "DIAMETRE",
    "FABRICANT",
    "MODELE",
    "MODE_DE_RELEVE",
    "DATE_DEPOSE"
]

print("\nColonnes disponibles parmi celles recherchées :")
for colonne in colonnes_utiles:
    print(colonne, "->", colonne in parc.columns)

Nombre de compteurs dans le parc : 361399
Nombre de colonnes : 105

Colonnes disponibles parmi celles recherchées :
NUMERO_SERIE -> True
DIAMETRE -> True
FABRICANT -> True
MODELE -> True
MODE_DE_RELEVE -> True
DATE_DEPOSE -> True


In [37]:
# Préparer les clés de jointure
rejet_etudie["Matricule compteur"] = (
    rejet_etudie["Matricule compteur"]
    .astype(str)
    .str.strip()
    .str.upper()
)

parc["NUMERO_SERIE"] = (
    parc["NUMERO_SERIE"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Garder uniquement les informations utiles du parc
parc_reduit = parc[
    [
        "NUMERO_SERIE",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "MODE_DE_RELEVE",
        "DATE_DEPOSE"
    ]
].drop_duplicates(subset=["NUMERO_SERIE"])

# Jointure entre les rejets ACT Métier et le parc compteurs
analyse_enrichie = rejet_etudie.merge(
    parc_reduit,
    left_on="Matricule compteur",
    right_on="NUMERO_SERIE",
    how="left"
)

# Vérifier combien de rejets ont été retrouvés dans le parc
nombre_total = len(analyse_enrichie)
nombre_trouves = analyse_enrichie["NUMERO_SERIE"].notna().sum()

print("Nombre de rejets étudiés :", nombre_total)
print("Compteurs retrouvés dans le parc :", nombre_trouves)
print("Compteurs non retrouvés :", nombre_total - nombre_trouves)

# Afficher les informations qui pourraient expliquer le traitement
analyse_enrichie[
    [
        "Matricule compteur",
        "Résultat",
        "Traitement",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "MODE_DE_RELEVE",
        "DATE_DEPOSE"
    ]
]

Nombre de rejets étudiés : 37
Compteurs retrouvés dans le parc : 5
Compteurs non retrouvés : 32


,Matricule compteur,Résultat,Traitement,DIAMETRE,FABRICANT,MODELE,MODE_DE_RELEVE,DATE_DEPOSE
0,D22VA808304,Compteur incompatible - vérifier le type et di...,Le 22/07/26 AT,15.0,Actaris - Flonic - Schlumberger - Itron,WOLTMAG EF,Télé-Relevé,NaN
1,I19JA200893 F01WA370994,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,NaN,NaN,NaN,NaN,NaN
2,D23BA109693 ASST,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,NaN,NaN,NaN,NaN,NaN
3,H26TA606863,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,NaN,NaN,NaN,NaN,NaN
4,D21BA101816,Compteur incompatible - vérifier le type et di...,Le 22/07/26 correction modèle,15.0,Actaris - Flonic - Schlumberger - Itron,Micro Précis II Annulaire fileté,Télé-Relevé,NaN
5,H26TA616045,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,NaN,NaN,NaN,NaN,NaN
6,I19JB023230 F19JB023230,Compteur incompatible - vérifier le type et di...,Le 22/07/26 Correction fabricant,NaN,NaN,NaN,NaN,NaN
7,H22VA215272,Compteur incompatible - vérifier le type et di...,Le 22/07/26 correction modèle,15.0,Sappel - Diehl Metering - MID,DIVERS : Sans Compteur,Télé-Relevé,NaN
8,H26TA532074,Compteur incompatible - vérifier le type et di...,EC le 22/07/26 asso,NaN,NaN,NaN,NaN,NaN
9,H26TA532142,Compteur incompatible - vérifier le type et di...,EC le 22/07/26 asso,NaN,NaN,NaN,NaN,NaN


In [38]:
# Créer une clé PDS à partir de l'ACT Métier
# Exemple : 981131738876 -> 1131738876

rejet_etudie["PDS_recherche"] = (
    rejet_etudie["PDS"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str[2:]
)

# Nettoyer l'identifiant PDS du parc compteur
parc["ID_PDS_recherche"] = (
    parc["ID_PDS"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Vérifier quelques valeurs avant la jointure
print("Exemples ACT Métier :")
print(
    rejet_etudie[
        ["PDS", "PDS_recherche", "Matricule compteur"]
    ].head()
)

print("\nExemples Parc compteur :")
print(
    parc[
        ["ID_PDS", "ID_PDS_recherche", "NUMERO_SERIE"]
    ].head()
)

# Vérifier combien de PDS des 37 rejets existent dans le parc
pds_trouves = rejet_etudie["PDS_recherche"].isin(
    parc["ID_PDS_recherche"]
)

print("\n--- Résultat ---")
print("Nombre de rejets étudiés :", len(rejet_etudie))
print("PDS retrouvés dans le parc :", pds_trouves.sum())
print("PDS non retrouvés :", (~pds_trouves).sum())

Exemples ACT Métier :
              PDS PDS_recherche         Matricule compteur
24   988059036201    8059036201                D22VA808304
29   987259037098    7259037098  I19JA200893   F01WA370994
171  985499011164    5499011164         D23BA109693   ASST
214  983914509966    3914509966                H26TA606863
231  981286471712    1286471712                D21BA101816

Exemples Parc compteur :
       ID_PDS ID_PDS_recherche NUMERO_SERIE
0  2242466434       2242466434   CTR_FICTIF
1    27618925         27618925  H22VA239424
2   230818441        230818441  D19BA079538
3   280510077        280510077  H24VA473587
4   673153904        673153904  H24VA093590

--- Résultat ---
Nombre de rejets étudiés : 37
PDS retrouvés dans le parc : 36
PDS non retrouvés : 1


In [39]:
# ============================================================
# COMPARAISON ACT MÉTIER ↔ PARC COMPTEUR PAR LE PDS
# ============================================================

# 1. On travaille uniquement sur les rejets "Compteur incompatible"
rejets_incompatibles = donnees[
    donnees["Résultat"]
    .astype(str)
    .str.contains("compteur incompatible", case=False, na=False)
].copy()


# 2. Nettoyer le PDS ACT Métier
# Exemple : 981131738876 -> 1131738876
rejets_incompatibles["PDS_ACT_NETTOYE"] = (
    rejets_incompatibles["PDS"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str[2:]   # suppression des 2 premiers caractères
)


# 3. Nettoyer le PDS du parc compteur
parc["PDS_PARC_NETTOYE"] = (
    parc["ID_PDS"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)


# 4. Nettoyer les numéros de compteur pour pouvoir les comparer
rejets_incompatibles["COMPTEUR_ACT"] = (
    rejets_incompatibles["Matricule compteur"]
    .astype(str)
    .str.strip()
    .str.upper()
)

parc["COMPTEUR_PARC"] = (
    parc["NUMERO_SERIE"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# 5. Faire la jointure grâce au PDS
comparaison = rejets_incompatibles.merge(
    parc[
        [
            "PDS_PARC_NETTOYE",
            "ID_PDS",
            "COMPTEUR_PARC",
            "NUMERO_SERIE",
            "DIAMETRE",
            "FABRICANT",
            "MODELE"
        ]
    ],
    left_on="PDS_ACT_NETTOYE",
    right_on="PDS_PARC_NETTOYE",
    how="left"
)


# 6. Vérifier si le compteur ACT est identique au compteur du parc
comparaison["Comparaison compteur"] = "PDS non retrouvé"

pds_trouve = comparaison["ID_PDS"].notna()

comparaison.loc[
    pds_trouve &
    (comparaison["COMPTEUR_ACT"] == comparaison["COMPTEUR_PARC"]),
    "Comparaison compteur"
] = "Compteur identique"

comparaison.loc[
    pds_trouve &
    (comparaison["COMPTEUR_ACT"] != comparaison["COMPTEUR_PARC"]),
    "Comparaison compteur"
] = "Compteur différent"


# 7. Résumé
print("Nombre de rejets Compteur incompatible :", len(rejets_incompatibles))

print("\nRésultat de la comparaison :")
print(comparaison["Comparaison compteur"].value_counts())


# 8. Afficher le détail
comparaison[
    [
        "PDS",
        "PDS_ACT_NETTOYE",
        "ID_PDS",
        "Matricule compteur",
        "NUMERO_SERIE",
        "Comparaison compteur",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "Traitement"
    ]
]

Nombre de rejets Compteur incompatible : 37

Résultat de la comparaison :
Comparaison compteur
Compteur différent    31
Compteur identique     5
PDS non retrouvé       1
Name: count, dtype: int64


,PDS,PDS_ACT_NETTOYE,ID_PDS,Matricule compteur,NUMERO_SERIE,Comparaison compteur,DIAMETRE,FABRICANT,MODELE,Traitement
0,988059036201,8059036201,8.059036e+09,D22VA808304,D22VA808304,Compteur identique,15.0,Actaris - Flonic - Schlumberger - Itron,WOLTMAG EF,Le 22/07/26 AT
1,987259037098,7259037098,7.259037e+09,I19JA200893 F01WA370994,F01WA370994,Compteur différent,15.0,Wateau - Elster,Marly II,Le 22/07/26 SOP
2,985499011164,5499011164,5.499011e+09,D23BA109693 ASST,ASST,Compteur différent,0.0,INCONNU,DIVERS : Sans Compteur,Le 22/07/26 SOP
3,983914509966,3914509966,3.914510e+09,H26TA606863,E02MA906849,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,"810 , 820",Le 22/07/26 SOP
4,981286471712,1286471712,1.286472e+09,D21BA101816,D21BA101816,Compteur identique,15.0,Actaris - Flonic - Schlumberger - Itron,Micro Précis II Annulaire fileté,Le 22/07/26 correction modèle
5,983281016078,3281016078,3.281016e+09,H26TA616045,180836,Compteur différent,15.0,Actaris - Flonic - Schlumberger - Itron,FLODIS,Le 22/07/26 SOP
6,985913615738,5913615738,5.913616e+09,I19JB023230 F19JB023230,F19JB023230,Compteur différent,20.0,Wateau - Elster,610,Le 22/07/26 Correction fabricant
7,983279700446,3279700446,3.279700e+09,H22VA215272,H22VA215272,Compteur identique,15.0,Sappel - Diehl Metering - MID,DIVERS : Sans Compteur,Le 22/07/26 correction modèle
8,983104871962,3104871962,3.104872e+09,H26TA532074,E07IA557683,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,620 C,EC le 22/07/26 asso
9,983116371960,3116371960,3.116372e+09,H26TA532142,E08IA415714,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,620 C,EC le 22/07/26 asso


In [40]:
import sys
sys.path.append("..")

from src.feature_engineering import ajouter_variables_metier
from src.treatment_analysis import (
    analyser_traitements_historiques,
    mesurer_regularite_traitement
)

donnees_enrichies = ajouter_variables_metier(donnees)

analyse_traitements = analyser_traitements_historiques(
    donnees_enrichies
)

analyse_traitements

,Résultat,Traitement,Nombre de cas,Pourcentage
0,Absence informations dans l'intervention,EC le 24/07/26 annul,4,100.0
1,Affectation KO,ok SITR,1,100.0
3,Aucune trame n'a été trouvée dans la plage hor...,Voir NC,348,99.7
2,Aucune trame n'a été trouvée dans la plage hor...,Le 24/07/26 MAJ num cptr,1,0.3
4,Ce PDS est absent du patrimoine - Action métie...,En attente,16,84.2
5,Ce PDS est absent du patrimoine - Action métie...,Erreur affact en attene ODY,2,10.5
6,Ce PDS est absent du patrimoine - Action métie...,Le 24/07/26 MAJ cptr,1,5.3
7,Ce PDS est déjà associé - Changer le libellé d...,EC le 24/07/26 chgt émet,34,100.0
8,Ce PDS n'est pas associé - Maintenance non réa...,Annulation,11,55.0
9,Ce PDS n'est pas associé - Maintenance non réa...,EC le 22/07/26 asso,4,20.0


In [41]:
regularite = mesurer_regularite_traitement(
    donnees_enrichies
)

regularite

,Résultat,Traitement principal,Nombre,Taux traitement principal (%)
0,Absence informations dans l'intervention,EC le 24/07/26 annul,4,100.0
1,Affectation KO,ok SITR,1,100.0
4,Ce PDS est déjà associé - Changer le libellé d...,EC le 24/07/26 chgt émet,34,100.0
10,l'association porte sur un émetteur installé a...,EC le 24/07/26 chgt cptr,1,100.0
7,Fichier fabricant non reçu,Le 24/07/26 IT4US,11,100.0
2,Aucune trame n'a été trouvée dans la plage hor...,Voir NC,348,99.7
3,Ce PDS est absent du patrimoine - Action métie...,En attente,16,84.2
8,L'acte métier porte sur un émetteur différent ...,EC le 24/07/26 chgt émet,2,66.7
9,Poids d'impulsion vide - vérifier le diamètre ...,EC le 24/07/26 asso,6,66.7
5,Ce PDS n'est pas associé - Maintenance non réa...,Annulation,11,55.0


In [42]:
import pandas as pd

from src.feature_engineering import ajouter_variables_metier
from src.parc_enrichment import enrichir_avec_parc

# Charger le bon parc compteur
parc = pd.read_csv(
    "../data/raw/Parc_compteur.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

# Ajouter les variables métier aux ACT
actes_prepares = ajouter_variables_metier(donnees)

# Enrichissement par le parc
comparaison_parc = enrichir_avec_parc(
    actes_prepares,
    parc
)

print(
    comparaison_parc["STATUT_COMPARAISON"]
    .value_counts(dropna=False)
)

comparaison_parc[
    [
        "PDS",
        "PDS_ODYSSEE",
        "Matricule compteur",
        "NUMERO_SERIE",
        "STATUT_COMPARAISON",
        "DIAMETRE_DEDUIT",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "Résultat",
        "Traitement"
    ]
]

ImportError: cannot import name 'enrichir_avec_parc' from 'src.parc_enrichment' (c:\Users\PC\Downloads\teleleve_anomaly_poc\notebooks\..\src\parc_enrichment.py)

In [ ]:
cas_incompatibles = comparaison_parc[
    comparaison_parc["Résultat"]
    .astype(str)
    .str.contains(
        "compteur incompatible",
        case=False,
        na=False
    )
].copy()

cas_incompatibles[
    [
        "PDS",
        "Matricule compteur",
        "NUMERO_SERIE",
        "STATUT_COMPARAISON",
        "DIAMETRE_DEDUIT",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "Traitement"
    ]
]

In [43]:
import pandas as pd

# Chargement du parc compteur
chemin_parc = "../data/raw/Parc_compteur.csv"

parc = pd.read_csv(
    chemin_parc,
    sep=";",
    encoding="utf-8",
    low_memory=False
)

print("Nombre de lignes :", parc.shape[0])
print("Nombre de colonnes :", parc.shape[1])

print("\nColonnes utiles disponibles :")
colonnes_utiles = [
    "ID_PDS",
    "NUMERO_SERIE",
    "DIAMETRE",
    "FABRICANT",
    "MODELE",
    "MODE_DE_RELEVE",
    "DATE_DEPOSE"
]

for colonne in colonnes_utiles:
    print(colonne, ":", colonne in parc.columns)

Nombre de lignes : 361399
Nombre de colonnes : 105

Colonnes utiles disponibles :
ID_PDS : True
NUMERO_SERIE : True
DIAMETRE : True
FABRICANT : True
MODELE : True
MODE_DE_RELEVE : True
DATE_DEPOSE : True


In [44]:
# 1. On récupère uniquement les cas "Compteur incompatible"
cas_incompatibles = donnees[
    donnees["Résultat"]
    .astype(str)
    .str.contains("compteur incompatible", case=False, na=False)
].copy()

# 2. Fonction simple pour nettoyer les identifiants
def nettoyer_id(valeur):
    if pd.isna(valeur):
        return None

    valeur = str(valeur).strip()

    # Cas fréquent après lecture Excel : 123456.0
    if valeur.endswith(".0"):
        valeur = valeur[:-2]

    return valeur


# 3. Préparation du PDS côté ACT Métier
cas_incompatibles["PDS_ACT"] = cas_incompatibles["PDS"].apply(nettoyer_id)

# On retire le préfixe 98 lorsqu'il est présent
cas_incompatibles["PDS_RECHERCHE"] = cas_incompatibles["PDS_ACT"].apply(
    lambda x: x[2:] if x and x.startswith("98") else x
)


# 4. Préparation du PDS côté parc compteur
parc["PDS_PARC"] = parc["ID_PDS"].apply(nettoyer_id)


# 5. Vérification : est-ce que le PDS existe dans le parc ?
pds_du_parc = set(parc["PDS_PARC"].dropna())

cas_incompatibles["PDS_TROUVE"] = (
    cas_incompatibles["PDS_RECHERCHE"].isin(pds_du_parc)
)


# 6. Résultat
print("Nombre de cas Compteur incompatible :", len(cas_incompatibles))
print(
    "PDS retrouvés dans le parc :",
    cas_incompatibles["PDS_TROUVE"].sum()
)
print(
    "PDS non retrouvés :",
    (~cas_incompatibles["PDS_TROUVE"]).sum()
)

print("\nTaux de rapprochement :")
print(
    round(cas_incompatibles["PDS_TROUVE"].mean() * 100, 1),
    "%"
)

display(
    cas_incompatibles[
        [
            "PDS",
            "PDS_RECHERCHE",
            "Matricule compteur",
            "Résultat",
            "Traitement",
            "PDS_TROUVE"
        ]
    ]
)

Nombre de cas Compteur incompatible : 37
PDS retrouvés dans le parc : 36
PDS non retrouvés : 1

Taux de rapprochement :
97.3 %


,PDS,PDS_RECHERCHE,Matricule compteur,Résultat,Traitement,PDS_TROUVE
24,988059036201,8059036201,D22VA808304,Compteur incompatible - vérifier le type et di...,Le 22/07/26 AT,True
29,987259037098,7259037098,I19JA200893 F01WA370994,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,True
171,985499011164,5499011164,D23BA109693 ASST,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,True
214,983914509966,3914509966,H26TA606863,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,True
231,981286471712,1286471712,D21BA101816,Compteur incompatible - vérifier le type et di...,Le 22/07/26 correction modèle,True
237,983281016078,3281016078,H26TA616045,Compteur incompatible - vérifier le type et di...,Le 22/07/26 SOP,True
247,985913615738,5913615738,I19JB023230 F19JB023230,Compteur incompatible - vérifier le type et di...,Le 22/07/26 Correction fabricant,True
252,983279700446,3279700446,H22VA215272,Compteur incompatible - vérifier le type et di...,Le 22/07/26 correction modèle,True
273,983104871962,3104871962,H26TA532074,Compteur incompatible - vérifier le type et di...,EC le 22/07/26 asso,True
274,983116371960,3116371960,H26TA532142,Compteur incompatible - vérifier le type et di...,EC le 22/07/26 asso,True


In [45]:
# Colonnes du parc utiles pour notre analyse
parc_reduit = parc[
    [
        "PDS_PARC",
        "NUMERO_SERIE",
        "DIAMETRE",
        "FABRICANT",
        "MODELE",
        "MODE_DE_RELEVE",
        "DATE_DEPOSE"
    ]
].copy()

# Nettoyage du numéro de série du compteur dans le parc
parc_reduit["COMPTEUR_PARC"] = (
    parc_reduit["NUMERO_SERIE"]
    .apply(nettoyer_id)
    .str.upper()
)

# Nettoyage du matricule compteur côté ACT Métier
cas_incompatibles["COMPTEUR_ACT"] = (
    cas_incompatibles["Matricule compteur"]
    .apply(nettoyer_id)
    .str.upper()
)

# Rapprochement ACT Métier <-> Parc compteur grâce au PDS
comparaison = cas_incompatibles.merge(
    parc_reduit,
    left_on="PDS_RECHERCHE",
    right_on="PDS_PARC",
    how="left"
)

# Comparaison des numéros de compteur
comparaison["COMPTEURS_IDENTIQUES"] = (
    comparaison["COMPTEUR_ACT"] == comparaison["COMPTEUR_PARC"]
)

# Statut lisible
comparaison["STATUT_COMPARAISON"] = "Compteur différent"

comparaison.loc[
    comparaison["PDS_PARC"].isna(),
    "STATUT_COMPARAISON"
] = "PDS non retrouvé"

comparaison.loc[
    comparaison["PDS_PARC"].notna()
    & comparaison["COMPTEURS_IDENTIQUES"],
    "STATUT_COMPARAISON"
] = "Compteur identique"


# Résumé
print("Résultat de la comparaison :")
print(comparaison["STATUT_COMPARAISON"].value_counts(dropna=False))

print("\nNombre de lignes après rapprochement :", len(comparaison))


# Affichage détaillé
display(
    comparaison[
        [
            "PDS",
            "Matricule compteur",
            "NUMERO_SERIE",
            "STATUT_COMPARAISON",
            "DIAMETRE",
            "FABRICANT",
            "MODELE",
            "Traitement"
        ]
    ]
)

Résultat de la comparaison :
STATUT_COMPARAISON
Compteur différent    31
Compteur identique     5
PDS non retrouvé       1
Name: count, dtype: int64

Nombre de lignes après rapprochement : 37


,PDS,Matricule compteur,NUMERO_SERIE,STATUT_COMPARAISON,DIAMETRE,FABRICANT,MODELE,Traitement
0,988059036201,D22VA808304,D22VA808304,Compteur identique,15.0,Actaris - Flonic - Schlumberger - Itron,WOLTMAG EF,Le 22/07/26 AT
1,987259037098,I19JA200893 F01WA370994,F01WA370994,Compteur différent,15.0,Wateau - Elster,Marly II,Le 22/07/26 SOP
2,985499011164,D23BA109693 ASST,ASST,Compteur différent,0.0,INCONNU,DIVERS : Sans Compteur,Le 22/07/26 SOP
3,983914509966,H26TA606863,E02MA906849,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,"810 , 820",Le 22/07/26 SOP
4,981286471712,D21BA101816,D21BA101816,Compteur identique,15.0,Actaris - Flonic - Schlumberger - Itron,Micro Précis II Annulaire fileté,Le 22/07/26 correction modèle
5,983281016078,H26TA616045,180836,Compteur différent,15.0,Actaris - Flonic - Schlumberger - Itron,FLODIS,Le 22/07/26 SOP
6,985913615738,I19JB023230 F19JB023230,F19JB023230,Compteur différent,20.0,Wateau - Elster,610,Le 22/07/26 Correction fabricant
7,983279700446,H22VA215272,H22VA215272,Compteur identique,15.0,Sappel - Diehl Metering - MID,DIVERS : Sans Compteur,Le 22/07/26 correction modèle
8,983104871962,H26TA532074,E07IA557683,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,620 C,EC le 22/07/26 asso
9,983116371960,H26TA532142,E08IA415714,Compteur différent,15.0,Invensys - Socam - Pam - Sensus,620 C,EC le 22/07/26 asso


In [46]:
# Tableau croisé :
# statut de comparaison du compteur VS traitement historique

croisement = pd.crosstab(
    comparaison["STATUT_COMPARAISON"],
    comparaison["Traitement"],
    margins=True
)

display(croisement)

Traitement,EC le 22/07/26 asso,EC le 24/07/26 asso,Le 22/07/26 AT,Le 22/07/26 Correction fabricant,Le 22/07/26 SOP,Le 22/07/26 correction modèle,Le 22/07/26 correction num cptr,Le 24/07/26 SOP,Le 24/07/26 correction fabricant,All
STATUT_COMPARAISON,,,,,,,,,,
Compteur différent,14,8,0,1,5,0,1,1,1,31
Compteur identique,0,0,1,0,0,4,0,0,0,5
PDS non retrouvé,0,0,0,0,1,0,0,0,0,1
All,14,8,1,1,6,4,1,1,1,37


In [47]:
# Affichage détaillé pour comprendre chaque décision

display(
    comparaison[
        [
            "STATUT_COMPARAISON",
            "Matricule compteur",
            "NUMERO_SERIE",
            "DIAMETRE",
            "FABRICANT",
            "MODELE",
            "Scénario",
            "Prémonté",
            "Traitement"
        ]
    ].sort_values(
        ["STATUT_COMPARAISON", "Traitement"]
    )
)

,STATUT_COMPARAISON,Matricule compteur,NUMERO_SERIE,DIAMETRE,FABRICANT,MODELE,Scénario,Prémonté,Traitement
8,Compteur différent,H26TA532074,E07IA557683,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
9,Compteur différent,H26TA532142,E08IA415714,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
10,Compteur différent,H26TA532147,10784,15.0,Actaris - Flonic - Schlumberger - Itron,FLODIS,Association,Oui,EC le 22/07/26 asso
11,Compteur différent,H26TA532079,E08IA610201,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
12,Compteur différent,H26TA672303,E07IA450277,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
13,Compteur différent,H26TA532143,E08IA610305,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
14,Compteur différent,H26TA532141,E08IA610200,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso
15,Compteur différent,H26TA672304,50346,15.0,Actaris - Flonic - Schlumberger - Itron,FLODIS,Association,Oui,EC le 22/07/26 asso
16,Compteur différent,H26TA532146,E01MA805471,15.0,Invensys - Socam - Pam - Sensus,"810 , 820",Association,Oui,EC le 22/07/26 asso
17,Compteur différent,H26TA672394,E07IA580170,15.0,Invensys - Socam - Pam - Sensus,620 C,Association,Oui,EC le 22/07/26 asso


In [49]:
# ============================================================
# ANALYSE DU DIAMÈTRE
# Comparaison entre le diamètre déduit du matricule ACT
# et le diamètre enregistré dans le parc compteur
# ============================================================

# Correspondance métier issue de la macro existante
correspondance_diametre = {
    "A": 15,
    "B": 20,
    "D": 30,
    "E": 40,
    "F": 50,
    "G": 60,
    "H": 80,
    "I": 100,
    "J": 125,
    "K": 150,
    "L": 200,
    "M": 250,
    "N": 300,
    "O": 400,
    "P": 500,
    "U": 15,
    "V": 15,
    "X": 0
}


# Fonction permettant de déduire le diamètre
# à partir du 5e caractère du matricule compteur
def deduire_diametre(matricule):

    if pd.isna(matricule):
        return None

    # Nettoyage du matricule
    matricule = str(matricule).strip().upper()

    # Certains champs contiennent plusieurs informations.
    # On conserve le premier matricule.
    matricule = matricule.split()[0]

    # Vérification de la longueur
    if len(matricule) < 5:
        return None

    # Le 5e caractère correspond au code diamètre
    code_diametre = matricule[4]

    return correspondance_diametre.get(code_diametre)


# ------------------------------------------------------------
# 1. Calcul du diamètre à partir du matricule de l'ACT Métier
# ------------------------------------------------------------

comparaison["DIAMETRE_ACT_DEDUIT"] = (
    comparaison["Matricule compteur"]
    .apply(deduire_diametre)
)


# ------------------------------------------------------------
# 2. Conversion du diamètre provenant du parc compteur
# ------------------------------------------------------------

diametre_parc = pd.to_numeric(
    comparaison["DIAMETRE"],
    errors="coerce"
)


# ------------------------------------------------------------
# 3. Comparaison des deux diamètres
# ------------------------------------------------------------

comparaison["DIAMETRE_IDENTIQUE"] = pd.Series(
    comparaison["DIAMETRE_ACT_DEDUIT"] == diametre_parc,
    index=comparaison.index,
    dtype="boolean"
)


# ------------------------------------------------------------
# 4. Si le PDS n'est pas retrouvé dans le parc,
#    la comparaison du diamètre est impossible
# ------------------------------------------------------------

comparaison.loc[
    comparaison["PDS_PARC"].isna(),
    "DIAMETRE_IDENTIQUE"
] = pd.NA


# ------------------------------------------------------------
# 5. Résumé
# ------------------------------------------------------------

print("Comparaison des diamètres :")

print(
    comparaison["DIAMETRE_IDENTIQUE"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 6. Affichage détaillé
# ------------------------------------------------------------

display(
    comparaison[
        [
            "Matricule compteur",
            "DIAMETRE_ACT_DEDUIT",
            "DIAMETRE",
            "DIAMETRE_IDENTIQUE",
            "STATUT_COMPARAISON",
            "Traitement"
        ]
    ].sort_values(
        ["DIAMETRE_IDENTIQUE", "Traitement"]
    )
)

Comparaison des diamètres :
DIAMETRE_IDENTIQUE
True     34
False     2
<NA>      1
Name: count, dtype: Int64


,Matricule compteur,DIAMETRE_ACT_DEDUIT,DIAMETRE,DIAMETRE_IDENTIQUE,STATUT_COMPARAISON,Traitement
2,D23BA109693 ASST,15,0.0,False,Compteur différent,Le 22/07/26 SOP
35,H25UA003428,15,NaN,False,Compteur différent,Le 24/07/26 SOP
8,H26TA532074,15,15.0,True,Compteur différent,EC le 22/07/26 asso
9,H26TA532142,15,15.0,True,Compteur différent,EC le 22/07/26 asso
10,H26TA532147,15,15.0,True,Compteur différent,EC le 22/07/26 asso
11,H26TA532079,15,15.0,True,Compteur différent,EC le 22/07/26 asso
12,H26TA672303,15,15.0,True,Compteur différent,EC le 22/07/26 asso
13,H26TA532143,15,15.0,True,Compteur différent,EC le 22/07/26 asso
14,H26TA532141,15,15.0,True,Compteur différent,EC le 22/07/26 asso
15,H26TA672304,15,15.0,True,Compteur différent,EC le 22/07/26 asso


In [50]:
# ============================================================
# SYNTHÈSE DES VARIABLES MÉTIER ET DU TRAITEMENT
# ============================================================

def normaliser_traitement(valeur):
    if pd.isna(valeur):
        return "Non renseigné"

    texte = str(valeur).lower().strip()

    if "asso" in texte:
        return "Association"
    elif "sop" in texte:
        return "Support (SOP)"
    elif "correction modèle" in texte:
        return "Correction modèle"
    elif "correction fabricant" in texte:
        return "Correction fabricant"
    elif "correction num cptr" in texte:
        return "Correction numéro compteur"
    elif "at" in texte:
        return "AT"
    else:
        return "Autre"


comparaison["TRAITEMENT_NORMALISE"] = (
    comparaison["Traitement"]
    .apply(normaliser_traitement)
)


# Tableau de synthèse
synthese = (
    comparaison
    .groupby(
        [
            "STATUT_COMPARAISON",
            "DIAMETRE_IDENTIQUE",
            "TRAITEMENT_NORMALISE"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="NOMBRE_CAS")
    .sort_values(
        ["STATUT_COMPARAISON", "NOMBRE_CAS"],
        ascending=[True, False]
    )
)

display(synthese)

,STATUT_COMPARAISON,DIAMETRE_IDENTIQUE,TRAITEMENT_NORMALISE,NOMBRE_CAS
1,Compteur différent,True,Association,22
4,Compteur différent,True,Support (SOP),4
0,Compteur différent,False,Support (SOP),2
2,Compteur différent,True,Correction fabricant,2
3,Compteur différent,True,Correction numéro compteur,1
6,Compteur identique,True,Correction modèle,4
5,Compteur identique,True,AT,1
7,PDS non retrouvé,<NA>,Support (SOP),1


In [51]:
# ============================================================
# ANALYSE GLOBALE DES TRAITEMENTS PAR TYPE DE REJET
# ============================================================

# Copie des 673 rejets
analyse_globale = donnees.copy()


# Normalisation des traitements
def normaliser_traitement_global(valeur):
    if pd.isna(valeur):
        return "Non renseigné"

    texte = str(valeur).lower().strip()

    if "asso" in texte:
        return "Association"
    elif "it4us" in texte:
        return "IT4US"
    elif "sop" in texte:
        return "Support (SOP)"
    elif "correction modèle" in texte:
        return "Correction modèle"
    elif "correction fabricant" in texte:
        return "Correction fabricant"
    elif "correction num cptr" in texte:
        return "Correction numéro compteur"
    elif "annulation" in texte:
        return "Annulation"
    elif "voir nc" in texte:
        return "Voir NC"
    elif "at" in texte:
        return "AT"
    else:
        return "Autre"


analyse_globale["TRAITEMENT_NORMALISE"] = (
    analyse_globale["Traitement"]
    .apply(normaliser_traitement_global)
)


# Tableau : Résultat SITR x Traitement normalisé
tableau_global = pd.crosstab(
    analyse_globale["Résultat"],
    analyse_globale["TRAITEMENT_NORMALISE"],
    margins=True
)

display(tableau_global)

TRAITEMENT_NORMALISE,AT,Annulation,Association,Autre,Correction fabricant,Correction modèle,Correction numéro compteur,IT4US,Non renseigné,Support (SOP),Voir NC,All
Résultat,,,,,,,,,,,,
Absence Matricule Emetteur,0,0,0,0,0,0,0,0,2,0,0,2
Absence informations dans l'intervention,0,0,0,4,0,0,0,0,0,0,0,4
Affectation KO,0,0,0,1,0,0,0,0,0,0,0,1
Aucune trame n'a été trouvée dans la plage horaire autorisée,0,0,0,1,0,0,0,0,0,0,348,349
Ce PDS est absent du patrimoine - Action métier impossible - Contacter le support,18,0,0,1,0,0,0,0,0,0,0,19
Ce PDS est déjà associé - Changer le libellé d'intervention,0,0,0,34,0,0,0,0,2,0,0,36
Ce PDS n'est pas associé - Maintenance non réalisable,0,11,6,3,0,0,0,0,0,0,0,20
Compteur déjà associé sur un PDS - A vérifier,0,0,0,0,0,0,0,0,7,0,0,7
Compteur incompatible - vérifier le type et diamètre compteur (900002011),1,0,22,0,2,4,1,0,0,7,0,37


In [52]:
# ============================================================
# ANALYSE DES TRAITEMENTS CLASSÉS "AUTRE"
# ============================================================

traitements_autres = (
    analyse_globale[
        analyse_globale["TRAITEMENT_NORMALISE"] == "Autre"
    ]
    .groupby(["Résultat", "Traitement"])
    .size()
    .reset_index(name="NOMBRE_CAS")
    .sort_values("NOMBRE_CAS", ascending=False)
)

print("Nombre total de cas classés 'Autre' :")
print(traitements_autres["NOMBRE_CAS"].sum())

display(traitements_autres)

Nombre total de cas classés 'Autre' :
50


,Résultat,Traitement,NOMBRE_CAS
4,Ce PDS est déjà associé - Changer le libellé d...,EC le 24/07/26 chgt émet,34
0,Absence informations dans l'intervention,EC le 24/07/26 annul,4
5,Ce PDS n'est pas associé - Maintenance non réa...,EC le 22/07/26 chgt émet vu avec AJ,3
7,L'acte métier porte sur un émetteur différent ...,EC le 24/07/26 chgt émet,2
1,Affectation KO,ok SITR,1
2,Aucune trame n'a été trouvée dans la plage hor...,Le 24/07/26 MAJ num cptr,1
3,Ce PDS est absent du patrimoine - Action métie...,Le 24/07/26 MAJ cptr,1
6,L'acte métier porte sur un émetteur différent ...,Doublon,1
8,Poids d'impulsion vide - vérifier le diamètre ...,EC le 24/07/26 EAU PRO + PELORCE S,1
9,Poids d'impulsion vide - vérifier le diamètre ...,Le 24/07/26 correction diam 15,1


In [53]:
# ============================================================
# NORMALISATION AMÉLIORÉE DES TRAITEMENTS
# ============================================================

def normaliser_traitement_global(valeur):
    if pd.isna(valeur):
        return "Non renseigné"

    texte = str(valeur).lower().strip()

    # Association
    if "asso" in texte:
        return "Association"

    # IT4US
    elif "it4us" in texte:
        return "IT4US"

    # Support
    elif "sop" in texte:
        return "Support (SOP)"

    # Annulation
    elif "annulation" in texte or "annul" in texte:
        return "Annulation"

    # Changement émetteur
    elif "chgt émet" in texte:
        return "Changement émetteur"

    # Changement compteur
    elif "chgt cptr" in texte:
        return "Changement compteur"

    # Correction modèle
    elif "correction modèle" in texte:
        return "Correction modèle"

    # Correction fabricant
    elif "correction fabricant" in texte:
        return "Correction fabricant"

    # Correction / mise à jour numéro compteur
    elif "correction num cptr" in texte or "maj num cptr" in texte:
        return "Correction numéro compteur"

    # Mise à jour compteur
    elif "maj cptr" in texte:
        return "Mise à jour compteur"

    # Correction diamètre
    elif "correction diam" in texte:
        return "Correction diamètre"

    # Doublon
    elif "doublon" in texte:
        return "Doublon"

    # Cas déjà OK dans SITR
    elif "ok sitr" in texte:
        return "OK SITR"

    # Voir NC
    elif "voir nc" in texte:
        return "Voir NC"

    # AT
    elif "at" in texte:
        return "AT"

    # Cas non interprété
    else:
        return "Autre"


# Application aux 673 rejets
analyse_globale["TRAITEMENT_NORMALISE"] = (
    analyse_globale["Traitement"]
    .apply(normaliser_traitement_global)
)


# Vérification des catégories obtenues
resume_traitements = (
    analyse_globale["TRAITEMENT_NORMALISE"]
    .value_counts(dropna=False)
    .reset_index()
)

resume_traitements.columns = [
    "TRAITEMENT_NORMALISE",
    "NOMBRE_CAS"
]

display(resume_traitements)

,TRAITEMENT_NORMALISE,NOMBRE_CAS
0,Voir NC,348
1,Non renseigné,185
2,Changement émetteur,39
3,Association,34
4,AT,19
5,Annulation,15
6,IT4US,11
7,Support (SOP),7
8,Correction modèle,4
9,Correction fabricant,3


In [54]:
# ============================================================
# MESURE DE LA RÉGULARITÉ DES TRAITEMENTS PAR TYPE DE REJET
# ============================================================

# On conserve uniquement les lignes pour lesquelles
# un traitement historique est renseigné
cas_renseignes = analyse_globale[
    analyse_globale["TRAITEMENT_NORMALISE"] != "Non renseigné"
].copy()


# Nombre de cas par Résultat et Traitement normalisé
repartition = (
    cas_renseignes
    .groupby(["Résultat", "TRAITEMENT_NORMALISE"])
    .size()
    .reset_index(name="NOMBRE_CAS")
)


# Nombre total de cas renseignés pour chaque Résultat
repartition["TOTAL_REJET"] = (
    repartition
    .groupby("Résultat")["NOMBRE_CAS"]
    .transform("sum")
)


# Pourcentage représenté par chaque traitement
repartition["POURCENTAGE"] = (
    repartition["NOMBRE_CAS"]
    / repartition["TOTAL_REJET"]
    * 100
).round(1)


# On récupère le traitement historique le plus fréquent
traitement_principal = (
    repartition
    .sort_values(
        ["Résultat", "NOMBRE_CAS"],
        ascending=[True, False]
    )
    .groupby("Résultat")
    .first()
    .reset_index()
)


# Noms plus compréhensibles
traitement_principal = traitement_principal.rename(
    columns={
        "TRAITEMENT_NORMALISE": "TRAITEMENT_PRINCIPAL",
        "NOMBRE_CAS": "NOMBRE_TRAITEMENT_PRINCIPAL",
        "TOTAL_REJET": "NOMBRE_CAS_RENSEIGNES",
        "POURCENTAGE": "TAUX_TRAITEMENT_PRINCIPAL"
    }
)


# Nombre de traitements différents observés pour chaque rejet
nb_traitements = (
    repartition
    .groupby("Résultat")["TRAITEMENT_NORMALISE"]
    .nunique()
    .reset_index(name="NOMBRE_TRAITEMENTS_DIFFERENTS")
)


# Fusion
synthese_regularite = traitement_principal.merge(
    nb_traitements,
    on="Résultat",
    how="left"
)


# Affichage du plus stable au plus ambigu
synthese_regularite = synthese_regularite[
    [
        "Résultat",
        "NOMBRE_CAS_RENSEIGNES",
        "NOMBRE_TRAITEMENTS_DIFFERENTS",
        "TRAITEMENT_PRINCIPAL",
        "NOMBRE_TRAITEMENT_PRINCIPAL",
        "TAUX_TRAITEMENT_PRINCIPAL"
    ]
].sort_values(
    ["TAUX_TRAITEMENT_PRINCIPAL", "NOMBRE_CAS_RENSEIGNES"],
    ascending=[False, False]
)

display(synthese_regularite)

,Résultat,NOMBRE_CAS_RENSEIGNES,NOMBRE_TRAITEMENTS_DIFFERENTS,TRAITEMENT_PRINCIPAL,NOMBRE_TRAITEMENT_PRINCIPAL,TAUX_TRAITEMENT_PRINCIPAL
4,Ce PDS est déjà associé - Changer le libellé d...,34,1,Changement émetteur,34,100.0
7,Fichier fabricant non reçu,11,1,IT4US,11,100.0
0,Absence informations dans l'intervention,4,1,Annulation,4,100.0
1,Affectation KO,1,1,OK SITR,1,100.0
10,l'association porte sur un émetteur installé a...,1,1,Changement compteur,1,100.0
2,Aucune trame n'a été trouvée dans la plage hor...,349,2,Voir NC,348,99.7
3,Ce PDS est absent du patrimoine - Action métie...,19,2,AT,18,94.7
9,Poids d'impulsion vide - vérifier le diamètre ...,9,4,Association,6,66.7
8,L'acte métier porte sur un émetteur différent ...,3,2,Changement émetteur,2,66.7
6,Compteur incompatible - vérifier le type et di...,37,6,Association,22,59.5


In [55]:
# ============================================================
# BASELINE : RÈGLES MÉTIER À FORTE RÉGULARITÉ HISTORIQUE
# ============================================================

# Règles candidates identifiées dans les données historiques
regles_traitement = {
    "Ce PDS est déjà associé - Changer le libellé d'intervention":
        "Changement émetteur",

    "Fichier fabricant non reçu":
        "IT4US",

    "Aucune trame n'a été trouvée dans la plage horaire autorisée":
        "Voir NC"
}


def proposer_traitement_par_regle(resultat):
    """
    Propose un traitement uniquement lorsqu'une règle
    à forte régularité historique a été identifiée.
    """

    if pd.isna(resultat):
        return pd.NA

    return regles_traitement.get(str(resultat).strip(), pd.NA)


# Application aux 673 rejets
analyse_globale["TRAITEMENT_PROPOSE_REGLE"] = (
    analyse_globale["Résultat"]
    .apply(proposer_traitement_par_regle)
)


# Mode de décision
analyse_globale["MODE_DECISION"] = "À analyser"

analyse_globale.loc[
    analyse_globale["TRAITEMENT_PROPOSE_REGLE"].notna(),
    "MODE_DECISION"
] = "Règle métier"


# ============================================================
# MESURE DE LA COUVERTURE DE LA BASELINE
# ============================================================

nombre_total = len(analyse_globale)

nombre_regles = (
    analyse_globale["TRAITEMENT_PROPOSE_REGLE"]
    .notna()
    .sum()
)

taux_couverture = nombre_regles / nombre_total * 100


print("Nombre total de rejets :", nombre_total)
print("Rejets couverts par une règle :", nombre_regles)
print(f"Taux de couverture : {taux_couverture:.1f} %")


# Répartition des traitements proposés
display(
    analyse_globale[
        analyse_globale["TRAITEMENT_PROPOSE_REGLE"].notna()
    ]["TRAITEMENT_PROPOSE_REGLE"]
    .value_counts()
    .rename_axis("Traitement proposé")
    .reset_index(name="Nombre de cas")
)

Nombre total de rejets : 673
Rejets couverts par une règle : 396
Taux de couverture : 58.8 %


,Traitement proposé,Nombre de cas
0,Voir NC,349
1,Changement émetteur,36
2,IT4US,11
